# Red team run

Creates a red team eval container with three risk evaluators and a run with multi-turn attack strategies.
Persists run id and output items to `artifacts/`.

Source: https://learn.microsoft.com/azure/foundry/how-to/develop/run-ai-red-teaming-cloud

In [ ]:
import json as _json
import os, pathlib, json, time
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Discover the active azd env dynamically: prefer AZURE_ENV_NAME, else read
# the 'defaultEnvironment' from agent/.azure/config.json.
AGENT_DIR = pathlib.Path('..') / 'agent'
_azd_base = AGENT_DIR / '.azure'
_env_name = os.environ.get('AZURE_ENV_NAME', '')
if not _env_name:
    _config = _azd_base / 'config.json'
    if _config.exists():
        try:
            _env_name = _json.loads(_config.read_text()).get('defaultEnvironment', '')
        except Exception:
            _env_name = ''
if _env_name and (_azd_base / _env_name / '.env').exists():
    AZD_ENV = _azd_base / _env_name / '.env'
else:
    AZD_ENV = next(
        (p / '.env' for p in sorted(_azd_base.iterdir()) if p.is_dir() and (p / '.env').exists()),
        _azd_base / 'default' / '.env',
    )
if AZD_ENV.exists():
    load_dotenv(AZD_ENV)
load_dotenv()
endpoint = os.environ['AZURE_AI_PROJECT_ENDPOINT']
model_deployment = os.environ.get('AZURE_AI_MODEL_DEPLOYMENT_NAME') or os.environ['MODEL_DEPLOYMENT_NAME']
agent_name = os.environ.get('AZURE_AI_AGENT_NAME', 'agent-framework-agent-basic-responses')
tax_path = pathlib.Path('../artifacts/pre-staged-taxonomy.json')
tax = json.loads(tax_path.read_text())
print('azd env:    ', _env_name or '(fallback)')
print('taxonomy id:', tax['id'])
project = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
openai_client = project.get_openai_client()

taxonomy id: azureai://accounts/ai-account-eyfs7o7lvdq7e/projects/ai-project-aiobs-foundry-20260520/evaluationtaxonomies/agent-framework-agent-basic-responses-prohibited-actions/versions/1.0


In [ ]:
red_team = openai_client.evals.create(
    name='Demo red team agentic safety',
    data_source_config={'type': 'azure_ai_source', 'scenario': 'red_team'},
    testing_criteria=[
        {'type': 'azure_ai_evaluator', 'evaluator_name': 'builtin.prohibited_actions',
         'name': 'Prohibited Actions', 'evaluator_version': '1'},
        {'type': 'azure_ai_evaluator', 'evaluator_name': 'builtin.task_adherence',
         'name': 'Task Adherence', 'evaluator_version': '1',
         'initialization_parameters': {'deployment_name': model_deployment}},
        {'type': 'azure_ai_evaluator', 'evaluator_name': 'builtin.sensitive_data_leakage',
         'name': 'Sensitive Data Leakage', 'evaluator_version': '1'},
    ],
)
RED_TEAM_ID = red_team.id
print('red team id:', RED_TEAM_ID)

In [ ]:
eval_run = openai_client.evals.runs.create(
    eval_id=RED_TEAM_ID,
    name='Demo red team run',
    data_source={
        'type': 'azure_ai_red_team',
        'item_generation_params': {
            'type': 'red_team_taxonomy',
            'attack_strategies': ['Flip', 'Base64', 'IndirectJailbreak'],
            'num_turns': 3,
            'source': {'type': 'file_id', 'id': tax['id']},
        },
        'target': tax['target'],
    },
)
RUN_ID = eval_run.id
print('run id:', RUN_ID, 'initial status:', eval_run.status)

In [ ]:
deadline = time.time() + 30 * 60
while time.time() < deadline:
    run = openai_client.evals.runs.retrieve(run_id=RUN_ID, eval_id=RED_TEAM_ID)
    print(time.strftime('%H:%M:%S'), run.status)
    if run.status in ('completed', 'failed', 'canceled', 'cancelled'):
        break
    time.sleep(30)
print('final status:', run.status)

In [ ]:
items = list(openai_client.evals.runs.output_items.list(run_id=RUN_ID, eval_id=RED_TEAM_ID))
out = [i.as_dict() if hasattr(i, 'as_dict') else i for i in items]
artifacts = pathlib.Path('../artifacts')
artifacts.mkdir(exist_ok=True)
(artifacts / f'redteam_eval_output_items_{agent_name}.json').write_text(json.dumps(out, indent=2, default=str))
(artifacts / 'pre-staged-redteam-run.json').write_text(json.dumps({'eval_id': RED_TEAM_ID, 'run_id': RUN_ID, 'item_count': len(out), 'final_status': run.status}, indent=2))
print(f'Saved {len(out)} output items.')